# Features: one row per window

Owner: Madeleine. Audience: the team and the write-up.

This notebook turns each labeled 15-by-25 window into one row of numbers a model can use. `window_features` returns 79 columns per window, and `FEATURE_NAMES` lists them in order. Every feature is relative to the window itself, because recorders at one site can sit up to 31 dB apart. PCA, k-means, and the nearest-neighbor background estimate come later and will be fit inside each cross-validation fold.

**Prerequisites**
- This branch, `madeleine/features`.
- `data/labeled_windows.npz`, built by `scripts/build_labeled_windows.py`. 
- `Test.csv` in the repo root. It is used only to check the layout. It is never used to fit anything.

**Outline**
1. Setup and data location
2. Labeled windows
3. Features
4. Feature takeaways
5. Above-floor share and the baselines
6. Findings and open decisions

## 1. Setup and data location

In [1]:
# avoid using stale version of script
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys


# add `src/` to the path
ROOT = Path.cwd()
if not (ROOT / "src" / "uvnc").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uvnc.units import db_to_linear, linear_to_db 
from uvnc.windows import windows_from_frame 
from uvnc.features import window_features, FEATURE_NAMES,  _above_floor


LABELED = ROOT / "data" / "labeled_windows.npz"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})



## 2. Labeled windows

Our training windows: 87,752 mixtures, their vessel share, and the site they're from.

In [4]:
data = np.load(LABELED)
print(data.files)

['mixtures', 'shares', 'site_ids']


In [5]:
#get arrays 
mixtures = data['mixtures']
shares = data['shares']
site_ids = data['site_ids']

for key in data.files:
    print(f"{key}: {data[key].shape}")

mixtures: (87752, 15, 25)
shares: (87752,)
site_ids: (87752,)


In [6]:
# minute needs all 25 bands blank to be missing
missing_minutes = np.isnan(mixtures).all(axis=2).sum(axis=1)

pd.Series(missing_minutes).value_counts().sort_index()

2    87752
Name: count, dtype: int64

In [7]:
display(np.isnan(mixtures[:, :, 0]).all(axis=0))

array([False, False, False, False, False, False, False, False, False,
        True, False,  True, False, False, False])

In [8]:
display(pd.Series(shares).describe())

count    8.775200e+04
mean     2.936509e-01
std      3.521028e-01
min      9.441511e-09
25%      1.084547e-02
50%      7.637237e-02
75%      6.082307e-01
max      9.998537e-01
dtype: float64

In [9]:
display(pd.Series(site_ids).value_counts())

1     52134
5     16698
13     8867
7      3371
4      3168
2      2976
9       294
10      244
Name: count, dtype: int64

## 3. Features

Calls `window_features` to make 79 columns:
- Per band (25 per): peak minus median, range, and spread (standard deviation over minutes)
- Broadband (2): the window's total level per minute. Gives peak minus median and minute of the peak
- Missing minutes: how many whole minutes are blank
- Share above the floor: energy above the band's 20th percentile.

In [10]:
features = window_features(mixtures)

In [11]:
print(features.shape)
print(np.nanmin(features), np.nanmax(features))

(87752, 79)
0.0013248125652864928 74.67807574348848


In [12]:
print(len(FEATURE_NAMES))
print(FEATURE_NAMES[:3], FEATURE_NAMES[-3:])

79
['peak_minus_median_TOL_31.6', 'peak_minus_median_TOL_39.8', 'peak_minus_median_TOL_50.1'] ['broadband_peak_minute', 'missing_minutes', 'above_floor_share']


## 4. Feature takeaways

Every training window is missing exactly two minutes.

The broadband peaks at minute 8 about a third of the time. Then, the median share is 0.78 (compared to about 0.01 to 0.04 otherwise). So broadband peak can follow just the ship when it is louder. In this training set, minute 8 matters because that is when every ship peaks.

Windows that peak at minute 1 have the .01 median share which is the lowest. These could be background noise not actual ships.

In [13]:
print(features.shape, features.shape[1] == len(FEATURE_NAMES))

print(pd.Series(features[:, 77]).value_counts())#missing min
print(pd.Series(features[:, 76]).value_counts())#peak min

(87752, 79) True
2.0    87752
Name: count, dtype: int64
8.0     28707
1.0     16400
9.0      6551
7.0      6237
15.0     4358
11.0     3766
6.0      3669
13.0     3115
14.0     3078
2.0      3035
5.0      3032
4.0      2918
3.0      2886
Name: count, dtype: int64


In [14]:
display(pd.Series(shares).groupby(features[:, 76]).median())

1.0     0.010086
2.0     0.021134
3.0     0.018408
4.0     0.020545
5.0     0.024860
6.0     0.041397
7.0     0.181031
8.0     0.784837
9.0     0.173033
11.0    0.030400
13.0    0.021715
14.0    0.019983
15.0    0.026297
dtype: float64

## 5. Above floor share and the baselines

In [15]:
heuristic = features[:, 78]

def score(pred, truth):
    return 1 - np.sqrt(np.mean((pred - truth)**2))

constant = np.full_like(shares, shares.mean())

print("baseline:", score(constant, shares))
print("heuristic:", score(heuristic, shares))

baseline: 0.6478991955251535
heuristic: 0.48896854503125775


In [16]:
print("mean heuristic:", heuristic.mean(), " mean true share:", shares.mean())
print("correlation:", np.corrcoef(heuristic, shares)[0, 1])

mean heuristic: 0.6330033551037402  mean true share: 0.29365093284324734
correlation: 0.22501105027498186


The heuristic score is 0.489 which is well below the baseline 0.648. It overpredicts (mean 0.633 compared to 0.294). This correlates weakly with the true share (0.225). The current labels also track background loudness, and because every ship is identical, these features ignore absolute level. This will need to be reran when ships vary.

In [17]:
floor_50 = _above_floor(mixtures, percentile=50)
print("mean:", floor_50.mean(), " correlation:", np.corrcoef(floor_50, shares)[0, 1])
print("score:", score(floor_50, shares))

mean: 0.45233181599744704  correlation: -0.08168490201014939
score: 0.5142921193909051


The floor is probably too low so means normal background noise could be misinterpreted as vessel energy. Even a floor based on the median does not do better (mean 0.452 and correlation -0.082)